# SafeWatch — Convert Keras Model to ONNX

Run this notebook in **Google Colab** (free GPU/CPU).

This version is updated to resolve Keras 3 deserialization compatibility issues by using `tf-keras` (legacy Keras 2), redirecting internal Keras imports to `tf-keras`, loading the model with `compile=False` to bypass custom metrics errors, and saving as a standard TensorFlow SavedModel before converting.

**Steps:**
1. Upload `safewatch_fall_model.keras` or `safewatch_fall_model_enhanced.keras` from your local `models/` folder
2. Run all cells
3. Download the output `fall_model.onnx`
4. Place it in `frontend/public/models/fall_model.onnx`

In [ ]:
# 1. Install tf-keras (legacy Keras 2 wrapper) and tf2onnx dependencies
!pip install tf-keras tf2onnx onnx onnxruntime tensorflow -q

In [ ]:
# 2. Upload the Keras model file
from google.colab import files
print('Upload your safewatch_fall_model.keras or safewatch_fall_model_enhanced.keras file:')
uploaded = files.upload()
keras_filename = list(uploaded.keys())[0]
print(f'Uploaded: {keras_filename}')

In [ ]:
# 3. Load the model using tf_keras with compile=False and legacy import redirection
# Note: Keras 2 models save internal module paths like 'keras.src.engine.functional'.
# Under Keras 3 (TensorFlow 2.16+), these paths do not exist. We redirect them to tf_keras
# to allow seamless deserialization without modifying the model file itself.
import sys
import tensorflow as tf
import numpy as np
import os

# Setup legacy Keras module redirections in sys.modules
try:
    import tf_keras as tfk
    import tf_keras.src.engine.functional as tfk_functional
    import tf_keras.src.engine.sequential as tfk_sequential
    
    sys.modules['keras.src.engine.functional'] = tfk_functional
    sys.modules['keras.src.engine.sequential'] = tfk_sequential
    sys.modules['keras.src.engine'] = tfk.src.engine
    sys.modules['keras.src'] = tfk.src
    sys.modules['keras'] = tfk
    print("✓ Successfully redirected legacy Keras import paths to tf_keras.")
except Exception as redirect_err:
    print(f"Warning: Legacy import redirection failed: {redirect_err}")

# Load model with compile=False to bypass custom metrics (like f1_score_metric)
try:
    print("Using tf_keras to load legacy Keras 2 model (compile=False)...")
    model = tfk.models.load_model(keras_filename, compile=False)
except Exception as e:
    print(f"tf_keras loading failed: {e}. Trying standard tf.keras (compile=False)...")
    model = tf.keras.models.load_model(keras_filename, compile=False)

print('Model input shape:', model.input_shape)
print('Model output shape:', model.output_shape)
model.summary()

In [ ]:
# 4. Export Keras model to standard TensorFlow SavedModel and convert to ONNX
import shutil

saved_model_path = "safewatch_saved_model"
if os.path.exists(saved_model_path):
    shutil.rmtree(saved_model_path)

try:
    # Exporting as SavedModel removes Keras library dependencies during ONNX conversion
    model.save(saved_model_path, save_format='tf')
    print("✓ Model successfully saved as TensorFlow SavedModel!")
    
    # Convert SavedModel directory to ONNX via tf2onnx CLI (most robust method)
    !python -m tf2onnx.convert --saved-model safewatch_saved_model --output fall_model.onnx --opset 13
    print("✓ ONNX conversion complete via tf2onnx CLI!")
except Exception as e:
    print(f"SavedModel export/conversion failed: {e}")
    print("Attempting direct tf2onnx Keras conversion...")
    import tf2onnx
    import onnx
    
    input_signature = [
        tf.TensorSpec(model.inputs[0].shape, tf.float32, name='input')
    ]
    onnx_model, _ = tf2onnx.convert.from_keras(
        model,
        input_signature=input_signature,
        opset=13
    )
    onnx.save(onnx_model, 'fall_model.onnx')
    print("✓ ONNX conversion complete via direct tf2onnx Keras converter!")

In [ ]:
# 5. Verify the generated ONNX model
import onnxruntime as ort

try:
    sess = ort.InferenceSession('fall_model.onnx')
    input_name = sess.get_inputs()[0].name
    input_shape = sess.get_inputs()[0].shape
    print(f'ONNX Input Name: {input_name}')
    print(f'ONNX Input Shape: {input_shape}')
    
    # Run inference with a dummy input
    dummy_input = np.random.randn(1, input_shape[1]).astype(np.float32)
    out = sess.run(None, {input_name: dummy_input})
    print(f'Test inference output: {out[0][0]:.4f} (expected: probability between 0 and 1)')
    print('✓ ONNX model is verified and working correctly!')
except Exception as e:
    print(f'✗ ONNX verification failed: {e}')

In [ ]:
# 6. Download the generated ONNX model
from google.colab import files
files.download('fall_model.onnx')
print('Downloaded! Place this file at: frontend/public/models/fall_model.onnx')